In [1]:
import os
import glob
import json
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from src.utils.evaluation import predictions_from_file, selection_score, metrics_table

In [2]:
ROOT = "outputs/results"
OUT = "outputs/tables"
def load_results(split):
    paths = sorted(glob.glob(f"{ROOT}/**/{split}_results.pt", recursive=True))
    return {os.path.relpath(os.path.dirname(p), ROOT): predictions_from_file(p) for p in paths}

train, val = load_results("train"), load_results("validation")

In [3]:
for split, data in [("train", train), ("validation", val)]:
    pd.concat({run: metrics_table(df) for run, df in data.items()}, names=["run", "domain"]).round(4).to_csv(f"{OUT}/all_metrics_{split}.csv")

Tables to compare different values of alpha, with LSTM and BiLSTM

In [ ]:
def bal_acc_table(runs, data):
    return pd.DataFrame({label: metrics_table(data[r])["recall"] for label, r in runs.items()}).T


alpha_runs_lstm = {
    "0.1": "frame\\lstm_dann_a0.1",
    "0.2": "frame\\lstm_dann_a0.2",
    "0.3": "frame\\lstm_dann_a0.3",
    "0.5": "frame\\lstm_dann_a0.5",
    "0.75": "frame\\lstm_dann_a0.75",
    "1.0": "frame\\lstm_dann_a1.0"
} 

alpha_table_lstm = bal_acc_table(alpha_runs_lstm, val)
alpha_table_lstm.to_csv(f"{OUT}/alpha_comparison_lstm.csv")
print("LSTM")
display(alpha_table_lstm.round(3).style.highlight_max(subset=["domain average"], color="purple"))

alpha_runs_bilstm = {
    "0.1": "frame\\bilstm_dann_a0.1",
    "0.2": "frame\\bilstm_dann_a0.2",
    "0.3": "frame\\bilstm_dann_a0.3",
    "0.5": "frame\\bilstm_dann_a0.5",
    "0.75": "frame\\bilstm_dann_a0.75",
    "1.0": "frame\\bilstm_dann_a1.0"
} 

alpha_table_bilstm = bal_acc_table(alpha_runs_bilstm, val)
alpha_table_bilstm.to_csv(f"{OUT}/alpha_comparison_bilstm.csv")
print("BiLSTM")
display(alpha_table_bilstm.round(3).style.highlight_max(subset=["domain average"], color="purple"))

LSTM


,animal,music,soundscapes,speech,domain average,overall
0.1,0.631000,0.640000,0.746000,0.942000,0.740000,0.889000
0.2,0.747000,0.671000,0.746000,0.933000,0.774000,0.894000
0.3,0.597000,0.682000,0.697000,0.941000,0.729000,0.880000
0.5,0.585000,0.637000,0.656000,0.939000,0.704000,0.869000
0.75,0.577000,0.704000,0.582000,0.932000,0.699000,0.850000
1.0,0.634000,0.710000,0.721000,0.934000,0.750000,0.877000


BiLSTM


,animal,music,soundscapes,speech,domain average,overall
0.1,0.624000,0.624000,0.779000,0.926000,0.738000,0.887000
0.2,0.731000,0.711000,0.754000,0.943000,0.785000,0.899000
0.3,0.745000,0.736000,0.770000,0.932000,0.796000,0.895000
0.5,0.598000,0.706000,0.754000,0.946000,0.751000,0.890000
0.75,0.602000,0.711000,0.697000,0.918000,0.732000,0.859000
1.0,0.659000,0.613000,0.746000,0.951000,0.742000,0.882000


Summary table for balanced accuracy for all approaches

In [10]:
summary = pd.DataFrame({
    r: {**{d: balanced_accuracy_score(g["y_true"], g["y_pred"]) for d, g in df.groupby("domain")},
        "mean": selection_score(df)}
    for r, df in val.items()
}).T

summary.round(3)

,animal,music,soundscapes,speech,mean
frame\bilstm,0.745,0.730,0.787,0.942,0.801
frame\bilstm_dann_a0.1,0.624,0.624,0.779,0.926,0.738
frame\bilstm_dann_a0.2,0.731,0.711,0.754,0.943,0.785
frame\bilstm_dann_a0.3,0.745,0.736,0.770,0.932,0.796
frame\bilstm_dann_a0.5,0.598,0.706,0.754,0.946,0.751
frame\bilstm_dann_a0.75,0.602,0.711,0.697,0.918,0.732
frame\bilstm_dann_a1.0,0.659,0.613,0.746,0.951,0.742
frame\bilstm_dann_a1.0_sampler,0.717,0.654,0.738,0.944,0.763
frame\bilstm_sampler,0.723,0.729,0.730,0.949,0.783
frame\lstm,0.586,0.590,0.762,0.936,0.719


Comparison between runs with and without weighted sampler

In [11]:
sampler_runs = {
    "LSTM_no_sampler": "frame\\lstm",
    "LSTM_with_sampler": "frame\\lstm_sampler",
    "BiLSTM_no_sampler": "frame\\bilstm",
    "BiLSTM_with_sampler": "frame\\bilstm_sampler"
}

sampler_table = bal_acc_table(sampler_runs, val)
sampler_table.round(3)

,animal,music,soundscapes,speech,domain average,overall
LSTM_no_sampler,0.586,0.590,0.762,0.936,0.719,0.881
LSTM_with_sampler,0.735,0.670,0.730,0.946,0.770,0.893
BiLSTM_no_sampler,0.745,0.730,0.787,0.942,0.801,0.910
BiLSTM_with_sampler,0.723,0.729,0.730,0.949,0.783,0.894


Balanced accuracy per domain, mean over domains and overall for all the final approaches (best alpha)

In [12]:
approach_runs = {
    "Utterance (with CLAP)": "utterance",
    "Utterance (with w2v2)": "utterance_w2v",
    "e2v only": "utterance_single\\e2v",
    "MERT only": "utterance_single\\mert",
    "AVES2 only": "utterance_single\\aves2",
    "CLAP only": "utterance_single\\clap",
    "LSTM": "frame\\lstm",
    "BiLSTM": "frame\\bilstm",
    "LSTM sampler": "frame\\lstm_sampler",
    "BiLSTM sampler": "frame\\bilstm_sampler",
    "LSTM + DANN": "frame\\lstm_dann_a0.2",
    "BiLSTM + DANN": "frame\\bilstm_dann_a0.3",

}

approach_table = bal_acc_table(approach_runs, val)
approach_table.to_csv(f"{OUT}/approach_comparison.csv")
approach_table.round(3)

,animal,music,soundscapes,speech,domain average,overall
Utterance (with CLAP),0.788,0.768,0.697,0.948,0.800,0.904
Utterance (with w2v2),0.784,0.732,0.730,0.948,0.799,0.898
e2v only,0.516,0.487,0.639,0.935,0.644,0.840
MERT only,0.550,0.856,0.820,0.680,0.727,0.697
AVES2 only,0.662,0.814,0.746,0.700,0.730,0.717
CLAP only,0.723,0.847,0.779,0.721,0.768,0.738
LSTM,0.586,0.590,0.762,0.936,0.719,0.881
BiLSTM,0.745,0.730,0.787,0.942,0.801,0.910
LSTM sampler,0.735,0.670,0.730,0.946,0.770,0.893
BiLSTM sampler,0.723,0.729,0.730,0.949,0.783,0.894
